In [ ]:
#w4
#Import Library
import pandas as pd                                 #จัดการข้อมูล (pandas)
from xgboost import XGBClassifier                   #โหลดโมเดล XGBoost
from sklearn.metrics import (
    f1_score,                                       #คํานวณ metric เช่น Accuracy, F1-score
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import json                                         
import os                                           

#กําหนด CONFIG
#NOTE: CONFIG
TEST_CSV  = r"F:\for-learning\Data\synthetic_plant_test.csv"                    #ไฟล์ข้อมูล Train
MODEL_IN  = r"F:\for-learning\Data\xgb_plant_model.json"                        #ไฟล์โมเดลทีเทรนแล้ว
META_OUT_JSON = r"F:\for-learning\Data\test_eval_metadata.json"         #ไฟล์ output สําหรับเก็บผลการประเมิน
META_OUT_CSV  = r"F:\for-learning\Data\test_eval_metadata.csv"

FEATURES = ["temp_c", "humidity_pct", "lux", "vpd_kpa"]                         #ตัวแปร inpu
TARGET = "y"                                                                    #ตัวแปรผลลัพธ์
LABELS = [0, 1, 2]                                                              #class ทีใช้ประเมิน

#โหลดข้อมูล TEST เทานั้น
#NOTE: Load TEST data only
test_df = pd.read_csv(TEST_CSV)                                                 #อ่านไฟล์ test dataset
test_df = test_df.dropna(subset=FEATURES + [TARGET])                            #ลบแถวทีมีค่า missing ใน feature หรือ target

X_test = test_df[FEATURES].values                                               #X_test → input features
y_test = test_df[TARGET].astype(int).values                                     #y_test → ค่าจริง (ground truth)

#โหลดโมเดลที่เทรนไว้แล้ว
#NOTE: Load model
model = XGBClassifier()                                                         #สร้าง object โมเดล
model.load_model(MODEL_IN)                                                      #โหลดนําหนัก (weights) จากไฟล์ .json
                                                                                # ไม่มีการ train ใหม่ เปนการใช้โมเดลทีเทรนไว้แล้ว
#ทํานาย (Prediction) บน Test Data                                                                                
#NOTE: Evaluate TEST
test_pred = model.predict(X_test)                                               #โมเดลทํานาย class ของแต่ละตัว

#คํานวณ Evaluation Metrics
test_f1 = f1_score(y_test, test_pred, average="macro", labels=LABELS)                               #Accuracy ความถูกต้องกี่%
test_acc = accuracy_score(y_test, test_pred)                                                        #Macro F1-score → ใช้กับ multi-class → ถ่วงนําหนักแต่ละ class เท่ากัน
conf_mat = confusion_matrix(y_test, test_pred, labels=LABELS).tolist()                              #Confusion Matrix เช็คจุดผิดพลาด
cls_report = classification_report(y_test, test_pred, labels=LABELS, digits=4, output_dict=True)    #Detailed classification report ระบุภาพรวมต่างๆออกมา

#สร้าง Metadata Dictionary
#NOTE: Save Metadata
metadata = {                                                                    #เก็บข้อมูล
    "test_samples": len(y_test),                                                #จํานวน test samples
    "test_class_distribution": test_df[TARGET].value_counts().to_dict(),        #การกระจาย class
    "accuracy": round(test_acc, 4),                                             #Accuracy
    "macro_f1": round(test_f1, 4),                                              #Macro-F1
    "confusion_matrix": conf_mat,                                               #Confusion matrix
    "classification_report": cls_report                                         #Classification report
}

#บันทึกผลลัพธ์
#JSON
with open(META_OUT_JSON, "w") as jf:                                            #ใช้สำหรับ
    json.dump(metadata, jf, indent=4)                                           #API,Dashboard backend,Model tracking system

#CSV                                                                            #ใช้สำหรับ
pd.DataFrame(cls_report).transpose().to_csv(META_OUT_CSV, index=True)           #เปดดูใน Excel,รายงาน,วิเคราะห์เพิมเติม

#แสดงผลใน Console
print("\n=== TEST PERFORMANCE (Never-seen data) ===")
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")
print("\nConfusion matrix:")
print(conf_mat)
print("\nClassification report:")
print(pd.DataFrame(cls_report).transpose())